# Direct Codegen: Exporting PyTorch & LiteRT Models for MATLAB/Simulink

Starting in **R2026a**, MATLAB can directly load and generate C/C++ code from **PyTorch ExportedProgram** (`.pt2`) and **LiteRT** (`.tflite`) models — no conversion to MATLAB network objects required. This tutorial walks you through the Python side: exporting models so they are ready for `loadPyTorchExportedProgram` and `loadLiteRTModel` in MATLAB.

We cover three progressively complex scenarios:

| # | Model | Difficulty | Export Format | Key Concept |
|---|-------|-----------|---------------|-------------|
| 1 | **RepViT** (image classifier) | Simple | PyTorch `.pt2` | One-shot export of a pretrained model |
| 2 | **YOLOv11** (object detector + segmentor) | Medium | LiteRT `.tflite` | Multi-output model (pre-exported) |
| 3 | **Stateful LSTM** (sensor activity recognition) | Advanced | PyTorch `.pt2` | Stateful model with explicit hidden state I/O |

**Prerequisites:** Basic Python familiarity. No PyTorch experience needed — every step is explained.

**How to use this notebook:** Click **Runtime > Run all**, or run each cell one by one.

---

## Setup: Install Packages

We create a **virtual environment** with **PyTorch 2.8.0** (CPU-only, the version supported for MATLAB codegen) and **timm**. The venv isolates us from Colab's pre-installed packages, avoiding version conflicts.

**Note:** After running the install cell, the notebook automatically switches to the venv's Python kernel. No manual restart needed.

In [ ]:
import subprocess, sys, os

# Create a virtual environment using virtualenv (venv may not be available on Colab)
venv_path = '/content/torch28_env'
if not os.path.exists(venv_path):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', venv_path], check=True)

pip = f'{venv_path}/bin/pip'
subprocess.run([pip, 'install', 'torch==2.8.0+cpu',
                '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([pip, 'install', '-q', 'timm', 'matplotlib', 'numpy'], check=True)

# Add venv site-packages to this process so subsequent cells use them
import site
site_pkgs = f'{venv_path}/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages'
site.addsitedir(site_pkgs)
sys.path.insert(0, site_pkgs)

print(f'Virtual environment ready at {venv_path}')

### Verify Installation

Let's confirm we have the correct PyTorch version and check if a GPU is available.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

assert torch.__version__.startswith('2.8.0'), \
    f"ERROR: Expected PyTorch 2.8.0 but got {torch.__version__}! " \
    f"Re-run the install cell above to set up the virtual environment."

---

# Part 1: RepViT — Image Classification (Simple)

[**RepViT**](https://arxiv.org/abs/2307.09283) (Rethinking Mobile ViT) is a family of lightweight vision models that incorporate efficient attention mechanisms from Vision Transformers into a MobileNet-style CNN backbone. The result is a model that runs fast on edge devices while still achieving strong accuracy on ImageNet classification (1000 object categories).

**Why this example?** RepViT is a great starting point because:
- It is a **single-input, single-output** model — the simplest case for `torch.export`
- It is available as a pretrained model in the popular [`timm`](https://github.com/huggingface/pytorch-image-models) library, so no training is needed
- It represents a realistic deployment scenario: a lightweight classifier suitable for mobile or embedded targets

In this section, we will:
1. Load a pretrained RepViT model from `timm`
2. Run a quick sanity check
3. Export it as a PyTorch ExportedProgram (`.pt2`) using `torch.export`

## 1.1 Load the Pretrained RepViT Model

We load the `repvit_m1_0` variant from `timm`. The tag `.dist_450e_in1k` means it was trained using knowledge distillation for 450 epochs on the ImageNet-1K dataset (1000 object classes).

In [ ]:
import timm

repvit_model = timm.create_model('repvit_m1_0.dist_450e_in1k', pretrained=True)
repvit_model.eval()

print(f'Model loaded: RepViT M1.0')
print(f'Number of parameters: {sum(p.numel() for p in repvit_model.parameters()):,}')

## 1.2 Sanity Check: Run a Forward Pass

RepViT expects RGB images at 224x224 resolution. The input tensor shape is `(batch, channels=3, height=224, width=224)`. The output is a vector of 1000 class scores — one per ImageNet category.

In [ ]:
sample_input = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    output = repvit_model(sample_input)

print(f'Input shape:  {sample_input.shape}')
print(f'Output shape: {output.shape}')

## 1.3 Export to PyTorch ExportedProgram (`.pt2`)

`torch.export` captures the model's **entire computation graph** — the mathematical operations, their order, and the weights. This is different from `torch.save()`, which only saves weights.

The exported `.pt2` file is what MATLAB uses for direct code generation via `loadPyTorchExportedProgram`.

In [ ]:
exported_repvit = torch.export.export(repvit_model, (sample_input,))

torch.export.save(exported_repvit, 'repvit.pt2')
print('Saved: repvit.pt2')
print(f'\nIn MATLAB, load this with:')
print('  net = loadPyTorchExportedProgram("repvit.pt2")')

That's it! Three lines of code: **load** the model, **export** the graph, **save** the file. Any PyTorch model that passes through `torch.export` can be loaded in MATLAB with `loadPyTorchExportedProgram`.

---

---

# Part 2: YOLOv11 — Object Detection & Segmentation (Medium)

[**YOLO**](https://docs.ultralytics.com/models/yolo11/) (You Only Look Once) is the most widely used family of real-time object detection models. Unlike two-stage detectors, YOLO processes the entire image in a **single forward pass**, making it extremely fast — often running at 30+ FPS even on modest hardware. **YOLOv11** is the latest generation from [Ultralytics](https://ultralytics.com/), with improved accuracy and a segmentation variant that outputs **pixel-level masks** alongside bounding boxes for 80 COCO object categories.

[**LiteRT**](https://ai.google.dev/edge/litert) (formerly TensorFlow Lite) is Google's runtime for deploying ML models on mobile, microcontrollers, and other edge devices. MATLAB can load LiteRT `.tflite` models using `loadLiteRTModel` and generate optimized C/C++ code for embedded targets.

**Why this example?** YOLOv11 is a step up in complexity because:
- The model has **multiple output tensors** — bounding boxes, class scores, and segmentation mask coefficients
- It demonstrates a different model format (`.tflite` instead of `.pt2`)
- The export pipeline (PyTorch → ONNX → TF → TFLite) requires specific package versions that conflict with Colab, so we show how to work with a **pre-exported** model

## 2.1 How Was This Model Exported?

The [Ultralytics](https://docs.ultralytics.com/) framework provides a one-line export from PyTorch to TFLite:

```python
from ultralytics import YOLO
model = YOLO('yolo11s-seg.pt')
model.export(format='tflite')
```

Internally this goes: **PyTorch → ONNX → TensorFlow SavedModel → TFLite**. Since this pipeline requires specific TensorFlow versions that conflict with Colab's environment, we download the pre-exported model directly from the [MATLAB Deep Learning GitHub repository](https://github.com/matlab-deep-learning/pretrained-yolo-v11-litert-model-for-segmentation-and-object-detection-with-matlab).

## 2.2 Download the Pre-Exported YOLOv11 LiteRT Model

The model is approximately 39 MB. It contains:
- **Input:** `[1, 640, 640, 3]` — a single RGB image resized to 640x640
- **Output 1:** `[1, 116, 8400]` — bounding boxes (4) + class scores (80) + mask coefficients (32) for up to 8400 candidate detections
- **Output 2:** `[1, 160, 160, 32]` — prototype segmentation masks

In [ ]:
import urllib.request
import os

url = 'https://github.com/matlab-deep-learning/pretrained-yolo-v11-litert-model-for-segmentation-and-object-detection-with-matlab/raw/main/yolo11s-seg_float32.tflite'
tflite_file = 'yolo11s-seg_float32.tflite'

if not os.path.exists(tflite_file):
    print('Downloading YOLOv11s-seg TFLite model (~39 MB)...')
    urllib.request.urlretrieve(url, tflite_file)

file_size_mb = os.path.getsize(tflite_file) / (1024 * 1024)
print(f'Downloaded: {tflite_file} ({file_size_mb:.1f} MB)')
print(f'\nIn MATLAB, load this with:')
print('  model = loadLiteRTModel("yolo11s-seg_float32.tflite")')

No Python export code was needed here — just a download. When a model is already available in `.tflite` format, you can use it directly with `loadLiteRTModel` in MATLAB.

---

---

# Part 3: Stateful LSTM — Sensor Activity Recognition (Advanced)

An [**LSTM**](https://en.wikipedia.org/wiki/Long_short-term_memory) (Long Short-Term Memory) is a recurrent neural network designed for **sequential data** — time series, sensor streams, text, and audio. Unlike feedforward networks that process each input independently, an LSTM maintains an internal **hidden state** that accumulates information across timesteps, giving it memory of what it has seen before.

In this section, we train a stateful LSTM on accelerometer data and export it to `.pt2` with **explicit hidden state inputs and outputs**.

### Why stateful?

In a real deployment scenario (e.g., a smartphone or microcontroller reading accelerometer data), sensor data arrives as a **continuous stream**. A stateful model carries its memory (hidden state) across inference calls:

```
Time 0: (sensor_data_0, h_init, c_init) → (prediction_0, h_1, c_1)
Time 1: (sensor_data_1, h_1,    c_1)    → (prediction_1, h_2, c_2)
Time 2: (sensor_data_2, h_2,    c_2)    → (prediction_2, h_3, c_3)
...
```

The hidden state `(h, c)` is an **explicit input and output** of the model — the caller manages it, not the model. This is the pattern required for Simulink integration, where a block receives one sample per timestep and must carry state across simulation steps.

**Why this example?** This is the most advanced case because:
- The model has **multiple inputs and outputs** — `(x, h_0, c_0) → (predictions, h_n, c_n)`
- We **train the model from scratch** on synthetic data, showing the full workflow
- The explicit state I/O pattern is essential for **Simulink deployment** and **streaming inference**

### The dataset

We use synthetic smartphone accelerometer data (3 axes: x, y, z) to classify 5 human activities: Dancing, Running, Sitting, Standing, and Walking. Each sample is 75 timesteps.

## 3.1 Generate Synthetic Accelerometer Data

We generate synthetic data that mimics real accelerometer patterns for each activity. Each activity has a distinct signature:
- **Sitting/Standing** — low amplitude, near-constant
- **Walking** — periodic, moderate amplitude
- **Running** — higher frequency and amplitude than walking
- **Dancing** — irregular, multi-frequency patterns

This keeps the tutorial self-contained — no external dataset downloads needed.

In [ ]:
import matplotlib.pyplot as plt

def generate_activity_data(num_samples_per_class=200, seq_len=75, num_features=3):
    activities = ['Dancing', 'Running', 'Sitting', 'Standing', 'Walking']
    X_all, Y_all = [], []

    for class_idx, activity in enumerate(activities):
        for _ in range(num_samples_per_class):
            t = np.linspace(0, 2 * np.pi, seq_len)
            noise = np.random.randn(seq_len, num_features) * 0.5

            if activity == 'Sitting':
                data = np.stack([np.ones(seq_len) * 0.1,
                                 np.ones(seq_len) * -9.8,
                                 np.ones(seq_len) * 0.05], axis=1) + noise * 0.3
            elif activity == 'Standing':
                data = np.stack([np.sin(t * 0.5) * 0.5,
                                 np.ones(seq_len) * -9.8,
                                 np.cos(t * 0.3) * 0.3], axis=1) + noise * 0.4
            elif activity == 'Walking':
                freq = np.random.uniform(1.5, 2.5)
                data = np.stack([np.sin(t * freq) * 3.0,
                                 np.sin(t * freq * 2) * 2.0 - 9.8,
                                 np.cos(t * freq) * 2.5], axis=1) + noise
            elif activity == 'Running':
                freq = np.random.uniform(3.0, 4.5)
                data = np.stack([np.sin(t * freq) * 8.0,
                                 np.sin(t * freq * 2) * 6.0 - 9.8,
                                 np.cos(t * freq) * 7.0], axis=1) + noise * 2.0
            elif activity == 'Dancing':
                f1, f2 = np.random.uniform(1, 3), np.random.uniform(2, 5)
                data = np.stack([np.sin(t * f1) * 5.0 + np.cos(t * f2) * 3.0,
                                 np.sin(t * f2) * 4.0 - 9.8,
                                 np.cos(t * f1) * 6.0 + np.sin(t * 3) * 2.0], axis=1) + noise * 1.5

            X_all.append(data)
            Y_all.append(class_idx)

    X = np.array(X_all, dtype=np.float32)
    Y = np.array(Y_all, dtype=np.int64)
    perm = np.random.permutation(len(X))
    return X[perm], Y[perm]

np.random.seed(42)
X_data, Y_data = generate_activity_data()
print(f'Dataset: {X_data.shape[0]} samples, {X_data.shape[1]} timesteps, {X_data.shape[2]} features')
print(f'Classes: Dancing(0), Running(1), Sitting(2), Standing(3), Walking(4)')

split = int(0.8 * len(X_data))
X_train, X_test = X_data[:split], X_data[split:]
Y_train, Y_test = Y_data[:split], Y_data[split:]

### Visualize the Accelerometer Patterns

Let's plot one sample per activity to see the difference in their accelerometer signatures.

In [ ]:
activities = ['Dancing', 'Running', 'Sitting', 'Standing', 'Walking']
fig, axes = plt.subplots(1, 5, figsize=(18, 3), sharey=True)

for i, activity in enumerate(activities):
    idx = np.where(Y_data == i)[0][0]
    axes[i].plot(X_data[idx])
    axes[i].set_title(activity)
    axes[i].set_xlabel('Timestep')
    if i == 0:
        axes[i].set_ylabel('Acceleration')
    axes[i].legend(['X', 'Y', 'Z'], fontsize=8)

plt.suptitle('Accelerometer Patterns by Activity', fontsize=14)
plt.tight_layout()
plt.show()

## 3.2 Define the Stateful LSTM Model

The key difference from a standard LSTM: **hidden state (`h`) and cell state (`c`) are explicit inputs and outputs** of the `forward()` method. This allows the caller to manage state between inference calls.

```
Standard:  forward(x)              → predictions
Stateful:  forward(x, h_0, c_0)    → (predictions, h_n, c_n)
```

Our model has:
- **LSTM layer** with 50 hidden units — learns temporal patterns in the accelerometer sequence
- **Fully connected layer** — maps each hidden state to 5 activity class scores

In [ ]:
class StatefulLSTM(nn.Module):
    def __init__(self, input_size=3, hidden_size=50, num_layers=1, num_classes=5):
        super(StatefulLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, h_0, c_0):
        lstm_out, (h_n, c_n) = self.lstm(x, (h_0, c_0))
        out = self.fc(lstm_out)
        return out, h_n, c_n

INPUT_SIZE = 3
HIDDEN_SIZE = 50
NUM_LAYERS = 1
NUM_CLASSES = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StatefulLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
print(model)
print(f'\nParameters: {sum(p.numel() for p in model.parameters()):,}')

## 3.3 Train the Model

During training, we initialize hidden states to zeros at the start of each sample. The model learns to update these states as it processes the 75-timestep sequence. We use the **last timestep's prediction** for classification.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(Y_test))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, Y_batch in train_loader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)
        batch_size = X_batch.size(0)

        h_0 = torch.zeros(NUM_LAYERS, batch_size, HIDDEN_SIZE, device=device)
        c_0 = torch.zeros(NUM_LAYERS, batch_size, HIDDEN_SIZE, device=device)

        optimizer.zero_grad()
        out, h_n, c_n = model(X_batch, h_0, c_0)
        last_pred = out[:, -1, :]
        loss = criterion(last_pred, Y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(last_pred, 1)
        total += Y_batch.size(0)
        correct += (predicted == Y_batch).sum().item()

    if (epoch + 1) % 5 == 0:
        acc = 100.0 * correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, Accuracy: {acc:.1f}%')

print('Training complete!')

### Evaluate on Test Data

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, Y_batch in test_loader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)
        batch_size = X_batch.size(0)

        h_0 = torch.zeros(NUM_LAYERS, batch_size, HIDDEN_SIZE, device=device)
        c_0 = torch.zeros(NUM_LAYERS, batch_size, HIDDEN_SIZE, device=device)

        out, _, _ = model(X_batch, h_0, c_0)
        last_pred = out[:, -1, :]
        _, predicted = torch.max(last_pred, 1)
        total += Y_batch.size(0)
        correct += (predicted == Y_batch).sum().item()

print(f'Test Accuracy: {100.0 * correct / total:.1f}%')

## 3.4 Export the Stateful LSTM to `.pt2`

Notice how the export includes **three inputs** `(x, h_0, c_0)` and **three outputs** `(predictions, h_n, c_n)`. This is what makes the model stateful — the caller is responsible for passing the hidden state from one call to the next.

In [ ]:
model_cpu = model.cpu()
model_cpu.eval()

sample_x = torch.randn(1, 75, 3)
sample_h0 = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)
sample_c0 = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)

exported_lstm = torch.export.export(
    model_cpu,
    (sample_x, sample_h0, sample_c0)
)

torch.export.save(exported_lstm, 'stateful_lstm.pt2')
print('Saved: stateful_lstm.pt2')
print(f'\nExported graph signature:')
print(f'  Inputs:  x {list(sample_x.shape)}, h_0 {list(sample_h0.shape)}, c_0 {list(sample_c0.shape)}')
print(f'  Outputs: predictions [1, 75, 5], h_n {list(sample_h0.shape)}, c_n {list(sample_c0.shape)}')

## 3.5 Verify: Simulate Stateful Inference

Let's verify the stateful pattern works correctly by processing a sequence **one timestep at a time**, carrying the hidden state between calls. This is exactly how the model will be used in a Simulink model receiving live sensor data.

We compare:
- **Method 1:** Feed the entire 75-step sequence at once
- **Method 2:** Feed one timestep at a time, passing `h_n, c_n` back as `h_0, c_0`

Both methods should produce the same final prediction.

In [ ]:
test_sample = torch.from_numpy(X_test[0:1])
true_label = Y_test[0]

model_cpu.eval()
with torch.no_grad():
    # Method 1: Full sequence at once
    h_0 = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)
    c_0 = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)
    full_out, _, _ = model_cpu(test_sample, h_0, c_0)
    full_pred = torch.argmax(full_out[0, -1, :]).item()

    # Method 2: One timestep at a time (stateful)
    h = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)
    c = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)
    for t in range(75):
        x_t = test_sample[:, t:t+1, :]
        out_t, h, c = model_cpu(x_t, h, c)
    step_pred = torch.argmax(out_t[0, -1, :]).item()

activities = ['Dancing', 'Running', 'Sitting', 'Standing', 'Walking']
print(f'True activity:        {activities[true_label]}')
print(f'Full-sequence pred:   {activities[full_pred]}')
print(f'Step-by-step pred:    {activities[step_pred]}')
print(f'\nBoth methods agree: {full_pred == step_pred}')

---

# Download All Exported Models

Run the cell below to download all model files to your local machine. You will use these in the MATLAB/Simulink portion of the tutorial.

In [ ]:
from google.colab import files
import os

export_files = [
    'repvit.pt2',
    'yolo11s-seg_float32.tflite',
    'stateful_lstm.pt2',
]

for f in export_files:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f'Downloading {f} ({size_mb:.1f} MB)...')
        files.download(f)
    else:
        print(f'Warning: {f} not found')

print('\nDone! You now have all models ready for MATLAB and Simulink.')

---

# Summary

| Model | Format | File | MATLAB Function | Difficulty |
|-------|--------|------|----------------|------------|
| RepViT (classifier) | PyTorch `.pt2` | `repvit.pt2` | `loadPyTorchExportedProgram` | Simple |
| YOLOv11 (detector + segmentor) | LiteRT `.tflite` | `yolo11s-seg_float32.tflite` | `loadLiteRTModel` | Medium |
| Stateful LSTM (activity recognition) | PyTorch `.pt2` | `stateful_lstm.pt2` | `loadPyTorchExportedProgram` | Advanced |

### Key Takeaways

1. **Simple models** — `torch.export.export()` + `torch.export.save()` is all you need
2. **Multi-output models** — frameworks like Ultralytics handle the conversion pipeline; pre-exported models can be downloaded directly
3. **Stateful models** — make hidden state explicit as model inputs/outputs so the caller manages state across inference calls

### What's Next? (Live Demo)

In the MATLAB/Simulink live demo, we will:
1. Load `repvit.pt2` with `loadPyTorchExportedProgram` and generate C/C++ code
2. Load `yolo11s-seg_float32.tflite` with `loadLiteRTModel` and run object detection
3. Load `stateful_lstm.pt2` with `loadPyTorchExportedProgram` for real-time activity classification